# Detector de colonias con YOLOv8, entrenado con AGAR

Cuaderno autocontenido para Google Colab. Entrena un detector de colonias con el
dataset publico AGAR y mide que tal funciona sobre las fotografias propias del
proyecto, tomadas con celular sobre un transiluminador.

## Que pregunta responde

El sistema actual, CellSAM con preprocesamiento, alcanza un acierto agregado del
80 % sobre las placas propias. Los experimentos de umbral y de mosaico mostraron
que esa via tiene techo, porque el limite esta en cuantas colonias propone el
detector y no en los filtros posteriores.

Ademas, una medicion morfologica con la escala controlada mostro que las
colonias propias son mas compactas, mas circulares y de borde mas nitido que las
de AGAR. Es decir, no son un caso mas dificil por su forma. Si eso es cierto, un
detector entrenado con AGAR deberia funcionar razonablemente bien sobre las
fotografias propias, y si no lo hace, la brecha estara en las condiciones de
captura y no en la biologia.

Este cuaderno mide exactamente eso.

## Antes de empezar

Activa la GPU en **Entorno de ejecucion, Cambiar tipo de entorno de ejecucion,
GPU**. Sin GPU el entrenamiento no es viable.

Sube `paquete_colab.zip` a tu Google Drive, en la raiz. Ese paquete contiene las
fotografias propias, el conteo manual de referencia, la muestra de AGAR y los
scripts necesarios.

## 1. Entorno y GPU

In [ ]:
import torch

print('PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('SIN GPU. Activala en Entorno de ejecucion antes de continuar.')

In [ ]:
!pip install --quiet ultralytics
from ultralytics import YOLO
import ultralytics
print('ultralytics', ultralytics.__version__)

## 2. Datos

Se monta Google Drive y se descomprime el paquete. Si prefieres subir el zip a
mano, sustituye la ruta por `/content/paquete_colab.zip` y omite el montaje.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile, os
from pathlib import Path

ZIP = Path('/content/drive/MyDrive/paquete_colab.zip')
BASE = Path('/content/trabajo')

if not ZIP.exists():
    raise SystemExit(f'No se encuentra {ZIP}. Subelo a la raiz de tu Drive.')

BASE.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(BASE)

for sub in sorted(os.listdir(BASE)):
    print(sub)

## 3. AGAR al formato YOLO

El script de conversion hace tres cosas que conviene recordar.

Pasa las cajas de coordenadas absolutas con origen en la esquina, que es como
las guarda AGAR, a coordenadas normalizadas con origen en el centro, que es lo
que espera YOLO.

Excluye las placas que AGAR marca con `colonies_number = -1`, que son las que
sus microbiologos consideraron incontables y vienen sin anotar. Entrenar con
ellas le enseñaria al modelo que una placa llena de colonias no tiene ninguna.

Junta las cinco especies en una sola clase llamada colonia, porque para contar no
hace falta distinguir especie y asi hay mas ejemplos por clase.

**Sobre el tamaño del conjunto.** La muestra libre de AGAR tiene 40 imagenes, de
las que unas 39 son utilizables. Sirve para comprobar que todo el flujo funciona,
pero no para entrenar un modelo util. El conjunto completo, 18.000 imagenes con
336.442 colonias, se consigue rellenando el formulario de registro en
https://agar.neurosys.com/ . Cuando lo tengas, sustituye la carpeta de origen por
la del conjunto completo y vuelve a ejecutar desde aqui.

In [ ]:
import subprocess

ORIGEN_AGAR = BASE / 'agar_muestra'      # cambia esto por el AGAR completo
SALIDA_YOLO = BASE / 'agar_yolo'

proceso = subprocess.run(
    ['python', str(BASE / 'scripts' / 'agar_a_yolo.py'),
     str(ORIGEN_AGAR), '--salida', str(SALIDA_YOLO)],
    capture_output=True, text=True)
print(proceso.stdout)
if proceso.returncode != 0:
    print('ERROR:', proceso.stderr)

In [ ]:
DATA_YAML = SALIDA_YOLO / 'data.yaml'
print(DATA_YAML.read_text())

## 4. Entrenamiento

Se parte de YOLOv8n preentrenado. Sus pesos vienen de imagenes generales que
nada tienen que ver con colonias, pero las primeras capas ya saben detectar
bordes y texturas, asi que converge mucho antes que empezando de cero.

Sobre los parametros. Se usa `imgsz=1280` en lugar del habitual 640 porque las
colonias son objetos pequeños dentro de una placa grande, y reducir demasiado la
imagen las hace desaparecer, que es justo el problema diagnosticado con CellSAM.
Se activan volteos y rotacion libre porque una placa no tiene orientacion
privilegiada, y se mantiene variacion de brillo y saturacion porque la
iluminacion de las fotografias propias difiere mucho de la de AGAR.

In [ ]:
modelo = YOLO('yolov8n.pt')

resultados = modelo.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=1280,
    batch=4,
    patience=20,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,      # variacion de brillo, clave por la diferencia de luz
    fliplr=0.5,
    flipud=0.5,
    degrees=180,    # una placa no tiene arriba ni abajo
    project='/content/runs',
    name='yolov8n_agar',
    exist_ok=True,
)

## 5. Rendimiento sobre AGAR

Primero se mide en el conjunto de prueba de AGAR, que el modelo no vio al
entrenar. Como referencia, los trabajos publicados sobre AGAR reportan mAP de
entre 76 % y 96 % segun la variante de YOLO y el subconjunto.

In [ ]:
metricas = modelo.val(split='test')
print(f'mAP50    : {metricas.box.map50:.3f}')
print(f'mAP50-95 : {metricas.box.map:.3f}')
print(f'precision: {metricas.box.mp:.3f}')
print(f'recall   : {metricas.box.mr:.3f}')

## 6. La prueba que importa: fotografias propias

Aqui se mide si un modelo entrenado con las imagenes limpias de AGAR sirve con
fotografias de celular sobre transiluminador.

Se comparan los conteos con el conteo manual de referencia de las 15 placas del
primer lote, hecho por la investigadora con anotacion por clic, y se calculan las
mismas metricas que se usaron con CellSAM para que la comparacion sea directa.

In [ ]:
import pandas as pd

FOTOS = BASE / 'mis_fotos'
GT = BASE / 'ground_truth' / 'ground_truth.csv'
TNTC = 250
MAX_DET = 500   # ver el comentario en la celda de prediccion

referencia = pd.read_csv(GT).set_index('image')['count'].to_dict()

filas = []
for ruta in sorted(FOTOS.glob('*.jpg')):
    # max_det alto: el valor por defecto de ultralytics es 300, y hay placas con
    # mas de 350 colonias, que quedarian topadas sin poder contarse enteras
    pred = modelo.predict(str(ruta), imgsz=1280, conf=0.25,
                          max_det=MAX_DET, verbose=False)[0]
    n = len(pred.boxes)
    manual = referencia.get(ruta.stem)
    filas.append({'placa': ruta.stem, 'manual': manual, 'yolo': n,
                  'error': (n - manual) if manual is not None else None})

comparacion = pd.DataFrame(filas)
comparacion

In [ ]:
val = comparacion.dropna(subset=['manual'])
contables = val[val['manual'] <= TNTC]

mae_cont = contables['error'].abs().mean()
mae_glob = val['error'].abs().mean()
acierto = 1 - val['error'].abs().sum() / val['manual'].sum()

print(f'YOLO entrenado con AGAR, evaluado sobre las fotografias propias')
print(f'  MAE rango contable : {mae_cont:.2f}')
print(f'  MAE global         : {mae_glob:.2f}')
print(f'  Acierto agregado   : {acierto*100:.1f}%')
print()
print('Sistema actual, CellSAM con preprocesamiento:')
print('  MAE rango contable 4.08, MAE global 11.33, acierto 80.1%')

### Como leer el resultado

Si el acierto se acerca o supera al 80 % del sistema actual, la conclusion es que
un detector entrenado con placas transfiere bien y no hace falta un modelo
propio, con lo que el aporte del trabajo se desplaza hacia las condiciones de
captura y el dataset.

Si el acierto cae mucho, existe una brecha de dominio real. Y como la medicion
morfologica descarto que se deba a la forma de las colonias, apuntaria a las
condiciones de imagen, es decir contraluz, punto quemado, viñeteo y balance de
blancos. Eso justificaria el ajuste con imagenes propias de la seccion
siguiente.

## 7. Ajuste con las fotografias propias

Las anotaciones ya existen, porque el conteo manual registro las coordenadas de
cada colonia. Como guarda puntos y no cajas, hay que estimar un tamaño; se usa
uno fijo tomado de la mediana observada, lo cual basta porque para contar
interesa acertar el centro mas que el contorno.

**Sobre la particion.** Una primera version separaba por serie, con M1 y M2 para
ajustar y el resto para validar. Resulto inservible, porque dejaba 29 colonias en
entrenamiento y 828 en validacion, con dos placas de entrenamiento
completamente vacias, y el entrenamiento divergio: la perdida de clasificacion
paso de 14 a 97 en siete epocas.

Aqui se reparte de otro modo. Las placas se ordenan por numero de colonias y se
van alternando entre entrenamiento y validacion, de manera que ambos lados
reciben placas escasas, medias y densas. Sigue siendo una particion honesta,
porque ninguna placa aparece en los dos lados, pero ya no enfrenta lo vacio
contra lo lleno.

**Advertencia que no cambia.** Con 15 placas el ajuste es un ensayo, no un
resultado. La medida limpia de rendimiento son las placas NRC73 del segundo
lote, que no participan ni en el ajuste ni en ninguna decision de parametros.

In [ ]:
import csv, shutil
from PIL import Image

PUNTOS = BASE / 'ground_truth'
PROPIO = BASE / 'mis_fotos_yolo'
LADO_CAJA = 60          # lado aproximado en pixeles de la imagen original

for sub in ['train', 'val']:
    (PROPIO / 'images' / sub).mkdir(parents=True, exist_ok=True)
    (PROPIO / 'labels' / sub).mkdir(parents=True, exist_ok=True)

# se ordena por densidad y se alterna, para que ambos lados tengan de todo
placas = []
for csv_p in sorted(PUNTOS.glob('*_puntos.csv')):
    stem = csv_p.name.replace('_puntos.csv', '')
    img_p = FOTOS / f'{stem}.jpg'
    if not img_p.exists():
        continue
    with open(csv_p, newline='') as f:
        pts = [(float(r['x']), float(r['y'])) for r in csv.DictReader(f)]
    placas.append((stem, img_p, pts))

placas.sort(key=lambda p: len(p[2]))
reparto = {stem: ('train' if i % 2 == 0 else 'val')
           for i, (stem, _, _) in enumerate(placas)}

n_train = n_val = 0
for stem, img_p, pts in placas:
    sub = reparto[stem]
    with Image.open(img_p) as im:
        W, H = im.size
    lineas = [f'0 {x/W:.6f} {y/H:.6f} {LADO_CAJA/W:.6f} {LADO_CAJA/H:.6f}'
              for x, y in pts]
    shutil.copy2(img_p, PROPIO / 'images' / sub / img_p.name)
    (PROPIO / 'labels' / sub / f'{stem}.txt').write_text('\n'.join(lineas))
    if sub == 'train':
        n_train += len(pts)
    else:
        n_val += len(pts)
    print(f'{stem:14} {sub:5} {len(lineas):4} colonias')

print(f'\nentrenamiento: {n_train} colonias, validacion: {n_val} colonias')

(PROPIO / 'data.yaml').write_text(
    f'path: {PROPIO.as_posix()}\n'
    'train: images/train\n'
    'val: images/val\n\n'
    'nc: 1\n'
    "names: ['colonia']\n")
print('Dataset propio listo')

In [ ]:
modelo_ajustado = YOLO('/content/runs/yolov8n_agar/weights/best.pt')

modelo_ajustado.train(
    data=str(PROPIO / 'data.yaml'),
    epochs=40,
    imgsz=1280,
    batch=4,
    lr0=0.001,      # tasa baja, para no borrar lo aprendido con AGAR
    project='/content/runs',
    name='yolov8n_ajustado',
    exist_ok=True,
)

## 8. Prueba ciega sobre el segundo lote

Las cuatro placas NRC73 no participaron en ningun ajuste ni en ninguna decision
de parametros, y su conteo manual todavia no se ha usado. Son por tanto el unico
conjunto que da una medida honesta de rendimiento sobre datos nuevos.

Esta celda produce los conteos. La comparacion con el conteo manual la hace la
investigadora, para preservar el caracter ciego de la prueba.

In [ ]:
LOTE2 = BASE / 'mis_fotos_lote2'

for ruta in sorted(LOTE2.glob('*.jpg')):
    for nombre, m in [('solo AGAR', modelo), ('ajustado', modelo_ajustado)]:
        pred = m.predict(str(ruta), imgsz=1280, conf=0.25,
                         max_det=MAX_DET, verbose=False)[0]
        print(f'{ruta.stem:16} {nombre:10} {len(pred.boxes):4} colonias')
    print()

## 9. Guardar los pesos

Conviene copiar los pesos a Drive, porque Colab borra el disco al cerrar la
sesion.

In [ ]:
import shutil
destino = Path('/content/drive/MyDrive/modelos_colonias')
destino.mkdir(exist_ok=True)
for nombre in ['yolov8n_agar', 'yolov8n_ajustado']:
    origen = Path(f'/content/runs/{nombre}/weights/best.pt')
    if origen.exists():
        shutil.copy2(origen, destino / f'{nombre}.pt')
        print('guardado', destino / f'{nombre}.pt')